# STimage Kidney Single-Slice Benchmark (INT18 -> INT21)

Strict STimage reproduction using this repository's HEST data and gene list, with 01-style outputs.

STimage's official model (`stimage/_model.py::CNN_NB_multiple_genes`) builds one Keras
output head per gene. At this repository's 17,511-gene panel that route does not
complete, which is measured and documented in `benchmark/hest_dataset.py`. This
notebook uses the source-equivalent feature-cached route: upstream's frozen
ResNet50/ImageNet no-top trunk plus GlobalAveragePooling2D is run once per spot and
cached, and upstream's per-gene `Dense(2)` heads are stacked into one einsum.

This is an exact reformulation, not an approximation -- the trunk is frozen
(`ft=False`) and a per-gene head is a linear map -- and `verify_stimage_vectorization`
checks it against upstream's own `negative_binomial_layer` and `negative_binomial_loss`
before any training happens. The row must be reported as **STimage (feature-cached)**.

Kept from upstream: the trunk, the negative-binomial parameterization, the
negative-binomial loss, Adam(1e-5) from the official `model.compile`, and
EarlyStopping on `val_loss` with `patience=10`.

Not kept, and therefore recorded: upstream augments training tiles every epoch
(`DataGenerator(aug=True)`), which a feature cache cannot do; and the target space is
this benchmark's shared `log1p(CPM-1e4)` rather than raw counts, so that every row
scores against one `true_bag`.

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Optional OpenSlide
try:
    import openslide
    HAS_OPENSLIDE = True
except ImportError:
    HAS_OPENSLIDE = False
    print("OpenSlide not found, using PIL fallback (may be slow).")

ROOT = Path(os.environ.get("MORPHO_VC_ROOT", "../")).expanduser().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "benchmark") not in sys.path:
    sys.path.insert(0, str(ROOT / "benchmark"))

import tensorflow as tf

from hest_dataset import (
    STIMAGE_FEATURE_DIM,
    STIMAGE_TILE_SIZE,
    bind_stimage_upstream,
    build_stimage_vectorized_head,
    extract_stimage_features,
    load_stimage_head_weights,
    save_stimage_head,
    stimage_nb_loss,
    stimage_spot_matrix,
    stimage_target_matrix,
    verify_stimage_vectorization,
)

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Single-slide protocol: train on TRAIN_SLIDE_ID, test on TEST_SLIDE_ID.
TRAIN_SLIDE_ID = "INT18"
TEST_SLIDE_ID = "INT21"
FOLD_TAG = "INT25_to_INT28"
all_ids = [TRAIN_SLIDE_ID, TEST_SLIDE_ID]

# Paths
STIMAGE_ROOT = ROOT / "benchmark" / "STimage"
result_dir = ROOT / "results" / "kidney_hest" / "stimage" / "single" / FOLD_TAG
result_dir.mkdir(parents=True, exist_ok=True)

feature_cache_dir = ROOT / "results" / "kidney_hest" / "stimage" / "features"
feature_cache_dir.mkdir(parents=True, exist_ok=True)

spatial_dir = ROOT / "data" / "spatial_data"
wsi_dir = ROOT / "data" / "hest_data" / "wsis"
common_gene_path = spatial_dir / "genesets" / "kidney_hest_v1_INT13_INT14_INT15_INT17_INT18_INT19_INT21_INT24.txt"

STIMAGE_WINDOW = STIMAGE_TILE_SIZE
STIMAGE_CROP_MODE = "tile_px"
STIMAGE_RESIZE_TO = STIMAGE_TILE_SIZE

print("Result dir:", result_dir)
print(f"Train: {TRAIN_SLIDE_ID} | Test: {TEST_SLIDE_ID}")
print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
print("Feature cache:", feature_cache_dir)


In [ ]:
# ===== Helpers =====

VAL_RATIO = 0.2
SPLIT_SEED = 42


def read_gene_list(path: Path):
    genes = path.read_text(encoding="utf-8").splitlines()
    return [g.strip() for g in genes if g.strip()]


def _compute_blocks(coords, grid_size):
    min_x, min_y = coords.min(axis=0)
    max_x, max_y = coords.max(axis=0)
    span_x = max_x - min_x + 1e-6
    span_y = max_y - min_y + 1e-6
    gx = np.floor((coords[:, 0] - min_x) / span_x * (grid_size - 1)).astype(int)
    gy = np.floor((coords[:, 1] - min_y) / span_y * (grid_size - 1)).astype(int)
    gx = np.clip(gx, 0, grid_size - 1)
    gy = np.clip(gy, 0, grid_size - 1)

    blocks = {}
    for i, (bx, by) in enumerate(zip(gx, gy)):
        bid = (int(bx), int(by))
        blocks.setdefault(bid, []).append(i)
    return blocks


def _select_grid_size(coords, min_grid=4, max_grid=10, min_blocks=20, max_blocks=80):
    target = (min_blocks + max_blocks) / 2.0
    candidates = []
    for g in range(min_grid, max_grid + 1):
        blocks = _compute_blocks(coords, g)
        n_blocks = len(blocks)
        in_range = min_blocks <= n_blocks <= max_blocks
        score = 0 if in_range else abs(n_blocks - target)
        candidates.append((in_range, score, g, blocks, n_blocks))
    candidates.sort(key=lambda x: (0 if x[0] else 1, x[1]))
    in_range, score, grid_size, blocks, n_blocks = candidates[0]
    return grid_size, blocks, n_blocks


def make_blocked_split(coords, val_ratio=0.2, seed=42, grid_size=None,
                       min_grid=4, max_grid=10, min_blocks=20, max_blocks=80):
    """Spatially blocked, quadrant-stratified validation split.

    Same logic the HisToGene / iStar / mclSTExp single-slide notebooks use, so
    the within-slide validation split is shared across methods.
    """
    n_spots = coords.shape[0]
    if grid_size is None:
        grid_size, blocks, block_count = _select_grid_size(
            coords,
            min_grid=min_grid,
            max_grid=max_grid,
            min_blocks=min_blocks,
            max_blocks=max_blocks,
        )
    else:
        blocks = _compute_blocks(coords, grid_size)
        block_count = len(blocks)

    block_sizes = {bid: len(idxs) for bid, idxs in blocks.items()}
    med_x, med_y = np.median(coords[:, 0]), np.median(coords[:, 1])

    quadrants = {0: [], 1: [], 2: [], 3: []}
    for bid, idxs in blocks.items():
        center = coords[idxs].mean(axis=0)
        qx = 0 if center[0] < med_x else 1
        qy = 0 if center[1] < med_y else 1
        q = qx + 2 * qy
        quadrants[q].append(bid)

    val_blocks = set()
    target_total = max(1, int(round(n_spots * val_ratio)))

    for q in range(4):
        block_ids = quadrants[q]
        if len(block_ids) < 2:
            continue
        rng_q = np.random.default_rng(seed + q + 1)
        block_ids = block_ids.copy()
        rng_q.shuffle(block_ids)

        total_q = sum(block_sizes[bid] for bid in block_ids)
        target_q = int(round(total_q * val_ratio))
        if target_q == 0:
            target_q = 1

        selected_q = []
        picked = 0
        for bid in block_ids:
            if len(selected_q) >= len(block_ids) - 1:
                break
            selected_q.append(bid)
            picked += block_sizes[bid]
            if picked >= target_q:
                break
        val_blocks.update(selected_q)

    val_count = sum(block_sizes[bid] for bid in val_blocks)
    if val_count < target_total:
        remaining = [bid for bid in blocks.keys() if bid not in val_blocks]
        remaining.sort(key=lambda b: block_sizes[b], reverse=True)
        for bid in remaining:
            if len(val_blocks) >= len(blocks) - 1:
                break
            val_blocks.add(bid)
            val_count += block_sizes[bid]
            if val_count >= target_total:
                break

    val_indices = []
    for bid in val_blocks:
        val_indices.extend(blocks[bid])
    val_indices = sorted(set(val_indices))
    val_set = set(val_indices)
    train_indices = [i for i in range(n_spots) if i not in val_set]

    actual_ratio = len(val_indices) / max(n_spots, 1)

    meta = {
        "strategy": "blocked_grid_quadrant",
        "grid_size": grid_size,
        "block_count": block_count,
        "val_ratio_target": val_ratio,
        "val_ratio_actual": actual_ratio,
        "min_grid": min_grid,
        "max_grid": max_grid,
        "seed": seed,
    }
    return train_indices, val_indices, meta


def pearson_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() <= 1:
        return np.nan
    a, b = a[m], b[m]
    if np.all(a == a[0]) or np.all(b == b[0]):
        return np.nan
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a * a).sum()) * np.sqrt((b * b).sum())
    return float((a * b).sum() / denom) if denom > 0 else np.nan


print("Helpers ready.")


In [ ]:
# ===== Step 1: Cache the frozen ResNet50-GAP trunk features (one 2048-d vector per spot) =====

genes = read_gene_list(common_gene_path)
print(f"Genes: {len(genes)}")

# Audit gate: confirm the vectorized head and loss reproduce upstream's own
# `negative_binomial_layer` / `negative_binomial_loss` before training on them.
bind_stimage_upstream(STIMAGE_ROOT)
vectorization_check = verify_stimage_vectorization(STIMAGE_ROOT)
print("Vectorization check:",
      f"head_rel={vectorization_check['relative_head_difference']:.2e}",
      f"loss_rel={vectorization_check['relative_loss_difference_float64']:.2e}")

feature_bundle = extract_stimage_features(
    all_ids,
    hest_dir=ROOT / "data" / "hest_data",
    spatial_dir=spatial_dir,
    window=STIMAGE_WINDOW,
    crop_mode=STIMAGE_CROP_MODE,
    resize_to=STIMAGE_RESIZE_TO,
    feature_cache_dir=feature_cache_dir,
    batch_size=128,
    verbose=True,
)

for sid in all_ids:
    print(f"{sid}: features {feature_bundle[sid]['shape']}")

with open(result_dir / "vectorization_check.json", "w", encoding="utf-8") as f:
    json.dump(vectorization_check, f, indent=2)


In [ ]:
# ===== Step 2: Targets + blocked within-slide validation split =====

import anndata as ad

targets = {}
coords_by_slide = {}

for sid in all_ids:
    adata = ad.read_h5ad(spatial_dir / f"{sid}.h5ad")
    mask = adata.obs["in_tissue"].to_numpy().astype(bool)
    targets[sid] = stimage_target_matrix(adata, genes)
    # (row, col) order used by the shared blocked-split helper
    coords_by_slide[sid] = adata.obs[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy()[mask][:, [1, 0]]
    print(f"{sid}: target {targets[sid].shape}")

train_sid = TRAIN_SLIDE_ID
train_idx, val_idx, meta = make_blocked_split(
    coords_by_slide[train_sid], val_ratio=VAL_RATIO, seed=SPLIT_SEED
)
print(f"\n[{train_sid}] train={len(train_idx)} val={len(val_idx)} "
      f"grid={meta['grid_size']} val_ratio={meta['val_ratio_actual']:.4f}")

train_feats, train_targets, _ = stimage_spot_matrix(
    {train_sid: feature_bundle[train_sid]}, [train_sid], targets)
X_train = np.asarray(train_feats[train_idx], dtype=np.float32)
Y_train = np.asarray(train_targets[train_idx], dtype=np.float32)
X_val = np.asarray(train_feats[val_idx], dtype=np.float32)
Y_val = np.asarray(train_targets[val_idx], dtype=np.float32)
print("X_train:", X_train.shape, "| X_val:", X_val.shape)


In [ ]:
# ===== Step 3: Train the vectorized NB head (upstream loss / optimizer / early stopping) =====

# Adam(1e-5) is what upstream `CNN_NB_multiple_genes` compiles with; batch 32 and
# patience 20 are upstream's own benchmark settings (`Fig2/benchmarking_her2st/
# stimage_her2st.py`: 299x299 tiles, batch 32, 100 epochs, EarlyStopping patience 20).
# MAX_EPOCHS is raised above upstream's 100 on purpose. At this repository's 17,511-gene
# panel the validation loss was still falling at epoch 100 -- a 100-epoch run selected
# its best epoch at 99, i.e. the final one, meaning the cap, not convergence, ended
# training. Early stopping is upstream's own stated stopping rule, so it is allowed to
# decide here and the epoch count is only a safety bound.
LEARNING_RATE = 1.0e-5
BATCH_SIZE = 32
MAX_EPOCHS = 1000
PATIENCE = 20

head = build_stimage_vectorized_head(STIMAGE_FEATURE_DIM, len(genes), seed=SEED)
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

def run_nb_epoch(head, optimizer, X, Y, batch_size, shuffle, seed):
    """One pass over (X, Y) using upstream STimage's objective.

    `optimizer=None` evaluates without updating, which is how validation is run.
    Loss and parameterization are upstream's; only the data layout differs (the
    trunk features are precomputed, see the notebook header).
    """
    order = np.arange(X.shape[0])
    if shuffle:
        np.random.default_rng(seed).shuffle(order)
    losses = []
    for start in range(0, len(order), batch_size):
        batch = order[start:start + batch_size]
        xb = tf.constant(X[batch])
        yb = tf.constant(Y[batch])
        if optimizer is None:
            losses.append(float(stimage_nb_loss(yb, head(xb))))
        else:
            with tf.GradientTape() as tape:
                loss = stimage_nb_loss(yb, head(xb))
            grads = tape.gradient(loss, head.trainable_variables)
            optimizer.apply_gradients(zip(grads, head.trainable_variables))
            losses.append(float(loss))
    return float(np.mean(losses)) if losses else float("nan")


def evaluate_and_save(pred_bag, true_bag, genes, out_dir):
    """Write pred_bag/true_bag/metrics.json in the shape the benchmark manifests read."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    pred_bag = np.asarray(pred_bag, dtype=np.float32)
    true_bag = np.asarray(true_bag, dtype=np.float32)
    mae = float(np.mean(np.abs(pred_bag - true_bag)))
    rmse = float(np.sqrt(np.mean((pred_bag - true_bag) ** 2)))
    corrs = np.array([pearson_corr(pred_bag[:, i], true_bag[:, i])
                      for i in range(pred_bag.shape[1])], dtype=np.float32)
    valid = np.where(np.isfinite(corrs))[0]
    mean_pearson = float(np.nanmean(corrs)) if valid.size else float("nan")
    best_idx = int(valid[np.argmax(corrs[valid])]) if valid.size else None
    best_gene = genes[best_idx] if best_idx is not None else "NA"

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Mean Pearson (gene-wise): {mean_pearson:.4f}")
    print(f"Best Gene: {best_gene} (Pearson={corrs[best_idx]:.4f})" if best_idx is not None else "Best Gene: NA")

    np.save(out_dir / "pred_bag.npy", pred_bag)
    np.save(out_dir / "true_bag.npy", true_bag)
    np.save(out_dir / "gene_corr_all.npy", corrs)
    with open(out_dir / "gene_names.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(str(g) for g in genes))
    with open(out_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump({
            "MAE": mae, "RMSE": rmse, "Mean_Pearson": mean_pearson,
            "Best_Gene": best_gene,
            "Best_Pearson": float(corrs[best_idx]) if best_idx is not None else None,
        }, f, ensure_ascii=False, indent=2)
    print("Saved pred_bag/true_bag/metrics.json ->", out_dir)
    return {"MAE": mae, "RMSE": rmse, "Mean_Pearson": mean_pearson}

best_val = float("inf")
best_epoch = -1
best_weights = None
epoch_since_best = 0
history = []

for epoch in range(MAX_EPOCHS):
    tl = run_nb_epoch(head, optimizer, X_train, Y_train, BATCH_SIZE, True, SEED + epoch)
    vl = run_nb_epoch(head, None, X_val, Y_val, BATCH_SIZE, False, SEED)
    history.append({"epoch": epoch, "train_loss": tl, "val_loss": vl})
    print(f"{epoch}, train_loss: {tl:.4f},\tvalid_loss: {vl:.4f}")

    if vl < best_val:
        best_val = vl
        best_epoch = epoch
        best_weights = [v.numpy().copy() for v in head.trainable_variables]
        epoch_since_best = 0
    else:
        epoch_since_best += 1

    if epoch_since_best >= PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}")
        break

head.kernel.assign(best_weights[0])
head.bias.assign(best_weights[1])
print(f"\nBest epoch {best_epoch} | valid NB loss {best_val:.4f} (restored)")

save_stimage_head(head, result_dir / "stimage_head.npz")
pd.DataFrame(history).to_csv(result_dir / "train_log.csv", index=False)
with open(result_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "STimage",
        "reported_as": "STimage (feature-cached)",
        "protocol": "single_slide_rotating",
        "train_slide_ids": [TRAIN_SLIDE_ID], "test_slide_ids": [TEST_SLIDE_ID],
        "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "seed": SEED,
        "window": STIMAGE_WINDOW, "crop_mode": STIMAGE_CROP_MODE,
        "resize_to": STIMAGE_RESIZE_TO, "feature_dim": STIMAGE_FEATURE_DIM,
        "cnn_base": "resnet50", "fine_tuning": False,
        "best_epoch": best_epoch, "best_valid_nb_loss": float(best_val),
        "n_genes": len(genes),
        "val_split": meta,
    }, f, ensure_ascii=False, indent=2)
print("Saved:", result_dir / "stimage_head.npz")


In [ ]:
# ===== Step 4: Evaluation + Save pred_bag/true_bag + metrics =====

# Upstream predicts the negative-binomial mean n*(1-p)/p, which is what
# `stimage/03_Prediction.py` obtains via `scipy.stats.nbinom.mean`.
test_feats, test_targets, _ = stimage_spot_matrix(
    {TEST_SLIDE_ID: feature_bundle[TEST_SLIDE_ID]}, [TEST_SLIDE_ID], targets)
pred_bag = np.asarray(
    head.nb_mean(tf.constant(np.asarray(test_feats, dtype=np.float32))).numpy(),
    dtype=np.float32,
)
true_bag = np.asarray(test_targets, dtype=np.float32)
print("Prediction Shape:", pred_bag.shape)

evaluate_and_save(pred_bag, true_bag, genes, result_dir)


In [ ]:
# ===== Step 5: Visualization (01-style, top-4 genes on the test slide) =====

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

VIS_ID = TEST_SLIDE_ID
adata_vis = ad.read_h5ad(spatial_dir / f"{VIS_ID}.h5ad")
mask_vis = adata_vis.obs["in_tissue"].to_numpy().astype(bool)
xy = adata_vis.obs[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy()[mask_vis]

pred_vals = pred_bag[: mask_vis.sum()]
true_vals = true_bag[: mask_vis.sum()]
corrs_vis = np.array([pearson_corr(pred_vals[:, i], true_vals[:, i]) for i in range(pred_vals.shape[1])])
valid_idx = np.where(np.isfinite(corrs_vis))[0]
top4 = valid_idx[np.argsort(corrs_vis[valid_idx])[::-1][:4]]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for j, gi in enumerate(top4):
    for row, (vals, tag) in enumerate([(true_vals, "measured"), (pred_vals, "predicted")]):
        ax = axes[row, j]
        sc = ax.scatter(xy[:, 0], xy[:, 1], c=vals[:, gi], s=3, cmap="viridis")
        ax.set_title(f"{genes[gi]} ({tag})\nr={corrs_vis[gi]:.3f}")
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(sc, ax=ax, fraction=0.046)
plt.tight_layout()
out_fig = result_dir / "top4_genes_vis.png"
plt.savefig(out_fig, dpi=150)
plt.close()
print("Saved:", out_fig)
